# Importação de Datasets Acadêmicos — FakeTrueBR

**Versão:** 1  
**Criado em:** 2026-05-24  

## Objetivo
Importar e padronizar o corpus FakeTrueBR para o schema CheckAI.  
Gera um arquivo `raw_padronizado` em `dados/pipeline_datasets_academicos/raw/faketruebr/`.

## Referência Acadêmica
Chavarro, J. P. et al. *FakeTrueBR: Um corpus brasileiro de notícias falsas.*  
In: Anais da Escola Regional de Banco de Dados (ERBD), 18ª ed., 2023, Palmas/PR.  
Disponível em: https://github.com/jpchav98/FakeTrue.Br

## Constraints
- `dataset_final_treino_v1.csv` e `dataset_final_treino_v2*.csv` não são lidos nem alterados
- Nenhum modelo é treinado
- Nenhum dataset curado existente é sobrescrito
- Este notebook gera apenas o arquivo `raw_padronizado` — curadoria em notebook separado
- Saída salva em `dados/pipeline_datasets_academicos/raw/faketruebr/` com timestamp

## Schema gerado
| Coluna | Descrição |
|---|---|
| `id_registro` | Identificador único (ex: FAKETRUEBR_000001) |
| `texto_principal` | Corpo do texto (fake ou true) |
| `label` | 0 = falso, 1 = verdadeiro |
| `label_detalhe` | FAKETRUEBR_FAKE ou FAKETRUEBR_REAL |
| `pipeline_origem` | datasets_academicos |
| `portal_origem` | Inferido da URL (Boatos.org, G1 Globo, etc.) |
| `origem_texto` | corpus_academico |
| `origem_qualidade` | ROTULO_ACADEMICO |
| `tamanho_chars` | len(texto_principal) |
| `data_publicacao` | (não disponível no corpus) |
| `url_origem` | URL original (link_f ou link_t) |
| `fonte_dataset` | FakeTrueBR |
| `referencia_dataset` | Citação completa |

## Estrutura do corpus original
O FakeTrueBR é um dataset de **pares alinhados**: cada linha contém uma notícia falsa (coluna `fake`) e sua correspondente verdadeira (coluna `true`). Este notebook desempilha (unpivot) os pares em registros individuais, gerando **2 entradas por linha** do CSV original (total esperado: 3.582 registros).

| Coluna original | Conteúdo |
|---|---|
| `title_fake` | Título/manchete da notícia falsa |
| `fake` | Texto completo da notícia falsa (Boatos.org) |
| `link_f` | URL da notícia falsa |
| `true` | Texto completo da notícia verdadeira (G1 Globo / Folha de S.Paulo) |
| `link_t` | URL da notícia verdadeira |

In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path


def _find_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'dados').is_dir():
        return cwd
    if (cwd.parent / 'dados').is_dir():
        return cwd.parent
    raise FileNotFoundError(
        f'Pasta dados nao encontrada em {Path.cwd()}. '
        'Execute o notebook a partir da raiz do projeto ou de src/.'
    )


PROJECT_ROOT = _find_project_root()

# --- Metadados do dataset ---
NOME_DATASET         = 'FakeTrueBR'
FONTE_DATASET        = 'FakeTrueBR'
PIPELINE_ORIGEM      = 'datasets_academicos'
ORIGEM_QUALIDADE     = 'ROTULO_ACADEMICO'
ORIGEM_TEXTO         = 'corpus_academico'
PREFIXO_ID           = 'FAKETRUEBR'

REFERENCIA_ACADEMICA = (
    'Chavarro, J. P. et al. '
    'FakeTrueBR: Um corpus brasileiro de noticias falsas. '
    'In: Anais da Escola Regional de Banco de Dados (ERBD), 18a ed., 2023, Palmas/PR. '
    'Disponivel em: https://github.com/jpchav98/FakeTrue.Br'
)

URL_GITHUB_MAIN      = 'https://raw.githubusercontent.com/jpchav98/FakeTrue.Br/main/FakeTrueBr_corpus.csv'
URL_GITHUB_MASTER    = 'https://raw.githubusercontent.com/jpchav98/FakeTrue.Br/master/FakeTrueBr_corpus.csv'
NOME_ARQUIVO_ORIGINAL = 'faketruebr_original.csv'

# --- Caminhos ---
PIPELINE_DIR    = PROJECT_ROOT / 'dados' / 'pipeline_datasets_academicos'
RAW_DIR         = PIPELINE_DIR / 'raw'
CURATED_DIR     = PIPELINE_DIR / 'curated'
FINAL_DIR       = PIPELINE_DIR / 'final'
RAW_DATASET_DIR = RAW_DIR / 'faketruebr'

# --- Schema de saída ---
COLUNAS_SCHEMA = [
    'id_registro', 'texto_principal', 'label', 'label_detalhe',
    'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade',
    'tamanho_chars', 'data_publicacao', 'url_origem',
    'fonte_dataset', 'referencia_dataset',
]

print(f'CWD         : {Path.cwd()}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'PIPELINE_DIR: {PIPELINE_DIR}')
print(f'RAW_DIR     : {RAW_DIR}')
print()
print(f'NOME_DATASET    : {NOME_DATASET}')
print(f'PIPELINE_ORIGEM : {PIPELINE_ORIGEM}')
print(f'ORIGEM_QUALIDADE: {ORIGEM_QUALIDADE}')
print()
print(f'URL_GITHUB_MAIN  : {URL_GITHUB_MAIN}')
print(f'URL_GITHUB_MASTER: {URL_GITHUB_MASTER}')

CWD         : C:\Users\offan\Desktop\ml-checkai\src
PROJECT_ROOT: C:\Users\offan\Desktop\ml-checkai
PIPELINE_DIR: C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos
RAW_DIR     : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw

NOME_DATASET    : FakeTrueBR
PIPELINE_ORIGEM : datasets_academicos
ORIGEM_QUALIDADE: ROTULO_ACADEMICO

URL_GITHUB_MAIN  : https://raw.githubusercontent.com/jpchav98/FakeTrue.Br/main/FakeTrueBr_corpus.csv
URL_GITHUB_MASTER: https://raw.githubusercontent.com/jpchav98/FakeTrue.Br/master/FakeTrueBr_corpus.csv


## Seção 1 — Download do FakeTrueBR

Tenta baixar `FakeTrueBr_corpus.csv` automaticamente do GitHub (branch `main`, depois `master`).

### Download manual (se o automático falhar)
1. Acesse: https://github.com/jpchav98/FakeTrue.Br
2. Baixe o arquivo `FakeTrueBr_corpus.csv`
3. Salve como: `dados/pipeline_datasets_academicos/raw/faketruebr/faketruebr_original.csv`
4. Execute a célula abaixo novamente.

> **Nota:** Se o arquivo já existir localmente, o download é pulado automaticamente.

In [2]:
RAW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
ARQUIVO_ORIGINAL = RAW_DATASET_DIR / NOME_ARQUIVO_ORIGINAL
STATUS_DOWNLOAD = None

if ARQUIVO_ORIGINAL.exists():
    print(f'Arquivo ja existe localmente — download pulado.')
    print(f'  {ARQUIVO_ORIGINAL}')
    STATUS_DOWNLOAD = 'ja_existe'
else:
    print('Tentando download automatico do GitHub...')
    for tentativa, url in enumerate([URL_GITHUB_MAIN, URL_GITHUB_MASTER], start=1):
        print(f'  [{tentativa}/2] {url}')
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                ARQUIVO_ORIGINAL.write_bytes(resp.content)
                print(f'  Download OK — {len(resp.content):,} bytes')
                print(f'  Salvo em: {ARQUIVO_ORIGINAL}')
                STATUS_DOWNLOAD = 'baixado'
                break
            else:
                print(f'  HTTP {resp.status_code} — tentando proxima URL...')
        except Exception as exc:
            print(f'  Erro de conexao: {exc}')

    if STATUS_DOWNLOAD != 'baixado':
        sep = '=' * 60
        print()
        print(sep)
        print('  DOWNLOAD AUTOMATICO FALHOU')
        print(sep)
        print('  Faca o download manual:')
        print('  1. Acesse: https://github.com/jpchav98/FakeTrue.Br')
        print('  2. Baixe: FakeTrueBr_corpus.csv')
        print(f'  3. Salve como: {ARQUIVO_ORIGINAL}')
        print('  4. Execute esta celula novamente.')
        print(sep)
        STATUS_DOWNLOAD = 'falhou'

print()
print(f'Status download: {STATUS_DOWNLOAD}')

Arquivo ja existe localmente — download pulado.
  C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\faketruebr\faketruebr_original.csv

Status download: ja_existe


## Seção 2 — Leitura e diagnóstico inicial

Lê o CSV original do FakeTrueBR e exibe informações sobre a estrutura bruta.  
Tenta múltiplos encodings (`utf-8`, `utf-8-sig`, `latin-1`) em sequência.

In [3]:
if not ARQUIVO_ORIGINAL.exists():
    raise FileNotFoundError(
        f'Arquivo nao encontrado: {ARQUIVO_ORIGINAL}. '
        'Execute a celula de download ou faca o download manual.'
    )

df_original = None
for enc in ['utf-8', 'utf-8-sig', 'latin-1']:
    try:
        df_original = pd.read_csv(ARQUIVO_ORIGINAL, encoding=enc)
        print(f'Lido com encoding: {enc}')
        break
    except UnicodeDecodeError:
        print(f'Encoding {enc} falhou, tentando proximo...')

assert df_original is not None, 'Nao foi possivel ler com nenhum encoding testado.'

print(f'Shape: {df_original.shape}')
print(f'Colunas: {list(df_original.columns)}')
print()
print('Dtypes:')
print(df_original.dtypes.to_string())
print()
print('Nulos por coluna:')
print(df_original.isnull().sum().to_string())

Lido com encoding: utf-8
Shape: (1791, 5)
Colunas: ['title_fake', 'fake', 'link_f', 'true', 'link_t']

Dtypes:
title_fake    object
fake          object
link_f        object
true          object
link_t        object

Nulos por coluna:
title_fake    0
fake          0
link_f        0
true          0
link_t        0


In [4]:
SEP = '=' * 65
print(SEP)
print('DIAGNOSTICO INICIAL — FakeTrueBR')
print(SEP)

n_pares = len(df_original)
print(f'Total de pares (fake + true): {n_pares}')
print(f'Total de registros esperados apos unpivot: {n_pares * 2}')

print()
print('--- Amostra (3 primeiros pares) ---')
for i, row in df_original.head(3).iterrows():
    print(f'[Par {i + 1}]')
    fake_txt = str(row.get('fake', ''))[:120]
    true_txt = str(row.get('true', ''))[:120]
    link_f   = str(row.get('link_f', ''))
    link_t   = str(row.get('link_t', ''))
    print(f'  fake  : {fake_txt}...')
    print(f'  link_f: {link_f}')
    print(f'  true  : {true_txt}...')
    print(f'  link_t: {link_t}')
    print()

print('--- Estatisticas de tamanho (colunas brutas) ---')
for col in ['fake', 'true']:
    if col in df_original.columns:
        tam = df_original[col].dropna().astype(str).str.len()
        print(f'Coluna {col!r}:')
        print(f'  min    : {tam.min()}')
        print(f'  max    : {tam.max()}')
        print(f'  media  : {tam.mean():.0f}')
        print(f'  mediana: {tam.median():.0f}')

print()
print('--- URLs de origem (samples) ---')
print('link_f (fake) — primeiros 3:')
for url in df_original['link_f'].dropna().head(3):
    print(f'  {url}')
print('link_t (true) — primeiros 3:')
for url in df_original['link_t'].dropna().head(3):
    print(f'  {url}')

DIAGNOSTICO INICIAL — FakeTrueBR
Total de pares (fake + true): 1791
Total de registros esperados apos unpivot: 3582

--- Amostra (3 primeiros pares) ---
[Par 1]
  fake  : carnaval em olinda. arrastão monstro. fazuele isso foi só um aperitivo do que tá por vir no carnaval país comandado por ...
  link_f: https://www.boatos.org/entretenimento/video-arrastao-olinda-pe-causa-lula-carnaval-2023.html
  true  :  circula pelas redes sociais um vídeo que mostra o expresidente luiz inácio lula da silva descendo de um jatinho no reci...
  link_t: https://g1.globo.com/pe/pernambuco/carnaval/2023/noticia/2023/02/20/video-homem-e-ameacado-com-arma-durante-assalto-e-leva-tapas-e-rasteira-a-caminho-do-carnaval-em-olinda.ghtml

[Par 2]
  fake  :  carro alegórico da escola de samba grande rio, que homenageava o diabo pegou fogo na abertura dos desfiles nesta sextaf...
  link_f: https://www.boatos.org/entretenimento/carro-alegorico-grande-rio-homenagem-diabo-pega-fogo-carnaval-2023.html
  true  :  pouco 

## Seção 3 — Conversão para schema CheckAI

O FakeTrueBR usa **pares alinhados**: cada linha do CSV contém uma notícia falsa e sua correspondente verdadeira. Este notebook faz o *unpivot* criando registros individuais.

**Regras de mapeamento:**

| Campo | Registro fake (label=0) | Registro real (label=1) |
|---|---|---|
| `texto_principal` | coluna `fake` | coluna `true` |
| `label` | 0 | 1 |
| `label_detalhe` | FAKETRUEBR_FAKE | FAKETRUEBR_REAL |
| `url_origem` | link_f | link_t |
| `portal_origem` | inferido de link_f | inferido de link_t |
| `data_publicacao` | `""` (não disponível) | `""` (não disponível) |
| `origem_qualidade` | ROTULO_ACADEMICO | ROTULO_ACADEMICO |

In [5]:
def inferir_portal(url: str) -> str:
    if not isinstance(url, str) or not url.strip():
        return 'Desconhecido'
    u = url.lower()
    if 'boatos.org' in u:
        return 'Boatos.org'
    if 'g1.globo' in u or 'g1.com' in u:
        return 'G1 Globo'
    if 'folha.uol' in u or 'folha.com' in u:
        return 'Folha de S.Paulo'
    if 'globo.com' in u:
        return 'Globo.com'
    return 'Desconhecido'


registros_fake = pd.DataFrame({
    'texto_principal':    df_original['fake'].values,
    'label':              0,
    'label_detalhe':      'FAKETRUEBR_FAKE',
    'pipeline_origem':    PIPELINE_ORIGEM,
    'portal_origem':      df_original['link_f'].apply(inferir_portal).values,
    'origem_texto':       ORIGEM_TEXTO,
    'origem_qualidade':   ORIGEM_QUALIDADE,
    'data_publicacao':    '',
    'url_origem':         df_original['link_f'].fillna('').values,
    'fonte_dataset':      FONTE_DATASET,
    'referencia_dataset': REFERENCIA_ACADEMICA,
})

registros_real = pd.DataFrame({
    'texto_principal':    df_original['true'].values,
    'label':              1,
    'label_detalhe':      'FAKETRUEBR_REAL',
    'pipeline_origem':    PIPELINE_ORIGEM,
    'portal_origem':      df_original['link_t'].apply(inferir_portal).values,
    'origem_texto':       ORIGEM_TEXTO,
    'origem_qualidade':   ORIGEM_QUALIDADE,
    'data_publicacao':    '',
    'url_origem':         df_original['link_t'].fillna('').values,
    'fonte_dataset':      FONTE_DATASET,
    'referencia_dataset': REFERENCIA_ACADEMICA,
})

df_unpivot = pd.concat([registros_fake, registros_real], ignore_index=True)

print(f'Registros apos unpivot:')
print(f"  label=0 (fake): {(df_unpivot['label'] == 0).sum()}")
print(f"  label=1 (real): {(df_unpivot['label'] == 1).sum()}")
print(f'  Total         : {len(df_unpivot)}')
print()
print('Distribuicao portal_origem por label:')
for lbl in [0, 1]:
    print(f'  label={lbl}:')
    print(df_unpivot[df_unpivot['label'] == lbl]['portal_origem'].value_counts().to_string())

Registros apos unpivot:
  label=0 (fake): 1791
  label=1 (real): 1791
  Total         : 3582

Distribuicao portal_origem por label:
  label=0:
portal_origem
Boatos.org    1791
  label=1:
portal_origem
G1 Globo            1533
Desconhecido         155
Folha de S.Paulo     103


## Seção 4 — Limpeza básica e geração de IDs

Aplica apenas limpeza mínima nesta etapa. A curadoria pesada acontece em `curadoria_datasets_academicos.ipynb`.

**Operações nesta etapa:**
1. Converter `texto_principal` para string
2. Strip de espaços no texto
3. Remover linhas com texto vazio (incluindo `'nan'`, `'None'`)
4. Remover linhas sem label válido (fora de {0, 1})
5. Calcular `tamanho_chars`
6. Gerar `id_registro` no formato `FAKETRUEBR_000001`
7. Reordenar colunas conforme schema

In [6]:
n_antes = len(df_unpivot)
df_clean = df_unpivot.copy()

# 1. Garantir tipo string
df_clean['texto_principal'] = df_clean['texto_principal'].astype(str)

# 2. Strip basico
df_clean['texto_principal'] = df_clean['texto_principal'].str.strip()

# 3. Remover texto vazio ou literal nan
mask_vazio = (
    df_clean['texto_principal'].isin(['', 'nan', 'None', 'NaN'])
    | df_clean['texto_principal'].isna()
)
n_vazio = mask_vazio.sum()
df_clean = df_clean[~mask_vazio].copy()

# 4. Remover sem label valido
mask_sem_label = ~df_clean['label'].isin([0, 1])
n_sem_label = mask_sem_label.sum()
df_clean = df_clean[~mask_sem_label].copy()

# 5. Calcular tamanho_chars
df_clean['tamanho_chars'] = df_clean['texto_principal'].str.len()

# 6. Gerar id_registro sequencial
df_clean = df_clean.reset_index(drop=True)
df_clean['id_registro'] = [
    f'{PREFIXO_ID}_{i + 1:06d}' for i in range(len(df_clean))
]
assert df_clean['id_registro'].is_unique, 'id_registro com duplicatas'

# 7. Ordenar colunas conforme schema
df_clean = df_clean[COLUNAS_SCHEMA]

n_depois = len(df_clean)

print('=== Limpeza basica ===')
print(f'Registros antes da limpeza : {n_antes}')
print(f'  Removidos (texto vazio)  : {n_vazio}')
print(f'  Removidos (sem label)    : {n_sem_label}')
print(f'Registros apos limpeza     : {n_depois}')
print()
print('Distribuicao por label:')
print(df_clean['label'].value_counts().rename({0: 'label=0 (fake)', 1: 'label=1 (real)'}).to_string())
print()
print('Estatisticas de tamanho_chars por label:')
for lbl in [0, 1]:
    s = df_clean[df_clean['label'] == lbl]['tamanho_chars']
    print(f'  label={lbl}: min={s.min()} | max={s.max()} | media={s.mean():.0f} | mediana={s.median():.0f}')
print()
print(f'Schema: {list(df_clean.columns)}')
print()
print('Amostra (2 fake + 2 real):')
amostra = pd.concat([
    df_clean[df_clean['label'] == 0].head(2),
    df_clean[df_clean['label'] == 1].head(2),
])
cols_display = ['id_registro', 'label', 'label_detalhe', 'portal_origem', 'tamanho_chars', 'texto_principal']
amostra_display = amostra[cols_display].copy()
amostra_display['texto_principal'] = amostra_display['texto_principal'].str[:100] + '...'
display(amostra_display)

=== Limpeza basica ===
Registros antes da limpeza : 3582
  Removidos (texto vazio)  : 0
  Removidos (sem label)    : 0
Registros apos limpeza     : 3582

Distribuicao por label:
label
label=0 (fake)    1791
label=1 (real)    1791

Estatisticas de tamanho_chars por label:
  label=0: min=121 | max=6973 | media=859 | mediana=650
  label=1: min=298 | max=32766 | media=3024 | mediana=2223

Schema: ['id_registro', 'texto_principal', 'label', 'label_detalhe', 'pipeline_origem', 'portal_origem', 'origem_texto', 'origem_qualidade', 'tamanho_chars', 'data_publicacao', 'url_origem', 'fonte_dataset', 'referencia_dataset']

Amostra (2 fake + 2 real):


,id_registro,label,label_detalhe,portal_origem,tamanho_chars,texto_principal
0,FAKETRUEBR_000001,0,FAKETRUEBR_FAKE,Boatos.org,496,carnaval em olinda. arrastão monstro. fazuele ...
1,FAKETRUEBR_000002,0,FAKETRUEBR_FAKE,Boatos.org,439,"carro alegórico da escola de samba grande rio,..."
1791,FAKETRUEBR_001792,1,FAKETRUEBR_REAL,G1 Globo,1005,circula pelas redes sociais um vídeo que mostr...
1792,FAKETRUEBR_001793,1,FAKETRUEBR_REAL,G1 Globo,1544,pouco antes de a beijaflor de nilópolis entrar...


## Seção 5 — Salvar arquivo raw padronizado

Salva o CSV padronizado (ainda **não curado**) em:

```
dados/pipeline_datasets_academicos/raw/faketruebr/faketruebr_raw_padronizado_YYYY-MM-DD_HH-MM-SS.csv
```

> **Importante:** Este arquivo **não está curado** para treino.  
> A curadoria (dedup, near-duplicate detection, overlap com V2, filtros de qualidade)  
> será realizada em `src/curadoria_datasets_academicos.ipynb`.

In [7]:
TS = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
ARQUIVO_SAIDA = RAW_DATASET_DIR / f"faketruebr_raw_padronizado_{TS}.csv"

df_clean.to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8')

print(f'Arquivo salvo: {ARQUIVO_SAIDA}')
print(f'Registros   : {len(df_clean)}')
print(f'Tamanho     : {ARQUIVO_SAIDA.stat().st_size:,} bytes')

Arquivo salvo: C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\faketruebr\faketruebr_raw_padronizado_2026-05-24_19-57-33.csv
Registros   : 3582
Tamanho     : 8,788,636 bytes


## Seção 6 — Relatório final

In [8]:
SEP = '=' * 65
print(SEP)
print('RELATORIO FINAL — importacao_datasets_academicos (FakeTrueBR)')
print(SEP)
print()
print(f'Dataset            : {NOME_DATASET}')
print(f'Pipeline           : {PIPELINE_ORIGEM}')
print(f'Origem qualidade   : {ORIGEM_QUALIDADE}')
print(f'Referencia         : {REFERENCIA_ACADEMICA}')
print()
print(f'Total de pares no corpus original : {n_pares}')
print(f'Total apos unpivot                : {n_antes}')
print(f'Total apos limpeza basica         : {n_depois}')
print(f'  Removidos (texto vazio)         : {n_vazio}')
print(f'  Removidos (sem label)           : {n_sem_label}')
print()
print('[Distribuicao por label]')
counts = df_clean['label'].value_counts().sort_index()
for lbl, cnt in counts.items():
    nome = 'fake (label=0)' if lbl == 0 else 'real (label=1)'
    print(f'  {nome}: {cnt}')
print()
print('[Tamanho texto por label]')
for lbl in [0, 1]:
    s = df_clean[df_clean['label'] == lbl]['tamanho_chars']
    print(f'  label={lbl}: media={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max()}')
print()
print('[Portal de origem]')
print(df_clean['portal_origem'].value_counts().to_string())
print()
print(f'Arquivo salvo: {ARQUIVO_SAIDA.name}')
print(f'Caminho      : {ARQUIVO_SAIDA}')
print()
AVISO = '!' * 65
print(AVISO)
print('AVISO: Este arquivo NAO esta curado para treino.')
print('Proxima etapa: src/curadoria_datasets_academicos.ipynb')
print('  - Deduplicacao interna e cross-dataset (overlap com V2)')
print('  - Near-duplicate detection')
print('  - Filtros de qualidade (textos curtos, conflito de label)')
print('  - Auditoria de proporcao e vies por portal')
print(AVISO)
print(SEP)

RELATORIO FINAL — importacao_datasets_academicos (FakeTrueBR)

Dataset            : FakeTrueBR
Pipeline           : datasets_academicos
Origem qualidade   : ROTULO_ACADEMICO
Referencia         : Chavarro, J. P. et al. FakeTrueBR: Um corpus brasileiro de noticias falsas. In: Anais da Escola Regional de Banco de Dados (ERBD), 18a ed., 2023, Palmas/PR. Disponivel em: https://github.com/jpchav98/FakeTrue.Br

Total de pares no corpus original : 1791
Total apos unpivot                : 3582
Total apos limpeza basica         : 3582
  Removidos (texto vazio)         : 0
  Removidos (sem label)           : 0

[Distribuicao por label]
  fake (label=0): 1791
  real (label=1): 1791

[Tamanho texto por label]


  label=0: media=859 | mediana=650 | min=121 | max=6973
  label=1: media=3024 | mediana=2223 | min=298 | max=32766

[Portal de origem]
portal_origem
Boatos.org          1791
G1 Globo            1533
Desconhecido         155
Folha de S.Paulo     103

Arquivo salvo: faketruebr_raw_padronizado_2026-05-24_19-57-33.csv
Caminho      : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\faketruebr\faketruebr_raw_padronizado_2026-05-24_19-57-33.csv

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
AVISO: Este arquivo NAO esta curado para treino.
Proxima etapa: src/curadoria_datasets_academicos.ipynb
  - Deduplicacao interna e cross-dataset (overlap com V2)
  - Near-duplicate detection
  - Filtros de qualidade (textos curtos, conflito de label)
  - Auditoria de proporcao e vies por portal
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


---

# Seção B — Fake.Br Corpus

**Dataset:** Fake.Br Corpus  
**Referência:** Monteiro et al., *Contributions to the Study of Fake News in Portuguese: New Corpus and Automatic Detection Results*, PROPOR 2018.  
**GitHub:** https://github.com/roneysco/Fake.br-Corpus  

## Estrutura do corpus
O Fake.Br Corpus é organizado em pastas separadas para notícias **fake** e **true**.  
Cada artigo é um arquivo `.txt` individual em `full_texts/fake/` e `full_texts/true/`.

## Outputs gerados por esta seção
- `dados/pipeline_datasets_academicos/raw/fakebr/fakebr_corpus_raw/` — corpus extraído
- `dados/pipeline_datasets_academicos/raw/fakebr/fakebr_raw_padronizado_YYYY-MM-DD_HH-MM-SS.csv`

## Constraints
- Nenhum arquivo FakeTrueBR é modificado
- Nenhum arquivo curated existente é sobrescrito
- Nenhum modelo é treinado

In [9]:
# --- Metadados Fake.Br Corpus ---
NOME_DATASET_FB      = 'Fake.Br Corpus'
FONTE_DATASET_FB     = 'Fake.Br Corpus'
PIPELINE_ORIGEM_FB   = 'datasets_academicos'
ORIGEM_QUALIDADE_FB  = 'ROTULO_ACADEMICO'
ORIGEM_TEXTO_FB      = 'corpus_academico'
PREFIXO_ID_FB        = 'FAKEBR'

REFERENCIA_ACADEMICA_FB = (
    'Monteiro, R. A. et al. '
    'Contributions to the Study of Fake News in Portuguese: '
    'New Corpus and Automatic Detection Results. '
    'In: Proceedings of the 13th International Conference on '
    'Computational Processing of Portuguese (PROPOR), 2018. '
    'Disponivel em: https://github.com/roneysco/Fake.br-Corpus'
)

URL_GITHUB_ZIP_MAIN   = 'https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/main.zip'
URL_GITHUB_ZIP_MASTER = 'https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/master.zip'

# --- Caminhos Fake.Br ---
RAW_FAKEBR_DIR = PIPELINE_DIR / 'raw' / 'fakebr'
RAW_FAKEBR_DIR.mkdir(parents=True, exist_ok=True)

print(f'RAW_FAKEBR_DIR  : {RAW_FAKEBR_DIR}')
print(f'NOME_DATASET    : {NOME_DATASET_FB}')
print(f'PREFIXO_ID      : {PREFIXO_ID_FB}')
print(f'ORIGEM_QUALIDADE: {ORIGEM_QUALIDADE_FB}')
print()
print(f'URL_ZIP_MAIN  : {URL_GITHUB_ZIP_MAIN}')
print(f'URL_ZIP_MASTER: {URL_GITHUB_ZIP_MASTER}')


RAW_FAKEBR_DIR  : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\fakebr
NOME_DATASET    : Fake.Br Corpus
PREFIXO_ID      : FAKEBR
ORIGEM_QUALIDADE: ROTULO_ACADEMICO

URL_ZIP_MAIN  : https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/main.zip
URL_ZIP_MASTER: https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/master.zip


## Seção B2 — Download do Fake.Br Corpus

Tenta baixar o repositório como ZIP do GitHub (branch `main`, depois `master`).

### Download manual (se o automático falhar)
1. Acesse: https://github.com/roneysco/Fake.br-Corpus
2. Clique em **Code → Download ZIP**
3. Extraia o conteúdo em: `dados/pipeline_datasets_academicos/raw/fakebr/fakebr_corpus_raw/`
4. Execute a célula abaixo novamente.

> **Estrutura esperada:** `full_texts/fake/fake-N.txt` e `full_texts/true/true-N.txt`

In [10]:
import zipfile
import io

FAKEBR_EXTRACT_DIR = RAW_FAKEBR_DIR / 'fakebr_corpus_raw'
STATUS_DOWNLOAD_FB = None

# Verificar se já extraído
if FAKEBR_EXTRACT_DIR.exists() and any(FAKEBR_EXTRACT_DIR.rglob('*.txt')):
    print(f'Corpus ja extraido localmente — download pulado.')
    print(f'  {FAKEBR_EXTRACT_DIR}')
    STATUS_DOWNLOAD_FB = 'ja_existe'
else:
    FAKEBR_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print('Tentando download do ZIP do GitHub...')
    baixado = False
    for tentativa, url in enumerate([URL_GITHUB_ZIP_MAIN, URL_GITHUB_ZIP_MASTER], start=1):
        print(f'  [{tentativa}/2] {url}')
        try:
            resp = requests.get(url, timeout=90)
            if resp.status_code == 200:
                print(f'  Download OK — {len(resp.content):,} bytes')
                with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
                    zf.extractall(FAKEBR_EXTRACT_DIR)
                print(f'  Extraido em: {FAKEBR_EXTRACT_DIR}')
                STATUS_DOWNLOAD_FB = 'baixado'
                baixado = True
                break
            else:
                print(f'  HTTP {resp.status_code}')
        except Exception as exc:
            print(f'  Erro: {exc}')

    if not baixado:
        SEP60 = '=' * 60
        print()
        print(SEP60)
        print('DOWNLOAD FALHOU — INSTRUCOES MANUAIS:')
        print(SEP60)
        print('1. Acesse: https://github.com/roneysco/Fake.br-Corpus')
        print('2. Clique em Code → Download ZIP')
        print(f'3. Extraia o conteudo em:')
        print(f'   {FAKEBR_EXTRACT_DIR}')
        print('   Estrutura esperada apos extracao:')
        print(f'   {FAKEBR_EXTRACT_DIR}/')
        print('   └── Fake.br-Corpus-master/')
        print('       └── full_texts/')
        print('           ├── fake/ (fake-1.txt ... fake-N.txt)')
        print('           └── true/ (true-1.txt ... true-N.txt)')
        print('4. Execute esta celula novamente.')
        print(SEP60)
        STATUS_DOWNLOAD_FB = 'falhou'

print(f'\nStatus download Fake.Br: {STATUS_DOWNLOAD_FB}')


Corpus ja extraido localmente — download pulado.
  C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\fakebr\fakebr_corpus_raw

Status download Fake.Br: ja_existe


In [11]:
def _encontrar_full_texts(base_dir: Path):
    """Localiza a pasta full_texts dentro do corpus extraído."""
    for candidate in sorted(base_dir.rglob('full_texts')):
        if candidate.is_dir():
            return candidate
    for candidate in sorted(base_dir.rglob('fake')):
        if candidate.is_dir() and any(candidate.glob('*.txt')):
            return candidate.parent
    return None


def _ler_txt(path: Path) -> str:
    """Lê arquivo .txt com fallback de encoding."""
    for enc in ['utf-8', 'utf-8-sig', 'latin-1']:
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    return ''


def _parse_meta(path: Path) -> dict:
    """
    Lê arquivo *-meta.txt do Fake.Br Corpus.
    Formato: linha1=autor, linha2=url, linha3=categoria, linha4=data, linhas5+=features
    """
    try:
        linhas = _ler_txt(path).strip().splitlines()
    except Exception:
        return {}
    out = {}
    if len(linhas) >= 2:
        out['url'] = linhas[1].strip() if linhas[1].strip().startswith('http') else ''
    if len(linhas) >= 3:
        out['categoria'] = linhas[2].strip()
    if len(linhas) >= 4:
        data_raw = linhas[3].strip()
        # Normalizar para ISO
        if data_raw and data_raw != 'None':
            import re as _re
            if _re.match(r'\d{4}-\d{2}-\d{2}', data_raw):
                out['data'] = data_raw[:10]
            elif _re.match(r'\d{2}/\d{2}/\d{4}', data_raw):
                p = data_raw.split('/')
                out['data'] = f'{p[2]}-{p[1]}-{p[0]}'
            else:
                out['data'] = ''
        else:
            out['data'] = ''
    return out


def _inferir_portal_de_url(url: str) -> str:
    """Infere portal de origem a partir da URL."""
    if not url:
        return 'DESCONHECIDO'
    u = url.lower()
    if 'folha.uol' in u or 'folha.com' in u:
        return 'Folha de S.Paulo'
    if 'g1.globo' in u:
        return 'G1 Globo'
    if 'estadao.com' in u or 'estado.com' in u:
        return 'Estadão'
    if 'veja.abril' in u:
        return 'Veja'
    if 'agenciabrasil' in u or 'agencia.brasil' in u:
        return 'Agência Brasil'
    if 'r7.com' in u:
        return 'R7'
    if 'uol.com' in u:
        return 'UOL'
    if 'terra.com' in u:
        return 'Terra'
    if 'correio' in u:
        return 'Correio'
    if 'ceticismopolitico' in u:
        return 'CeticismoPolítico'
    if 'boatos.org' in u:
        return 'Boatos.org'
    # Extrair domínio
    try:
        from urllib.parse import urlparse
        dom = urlparse(u).netloc.replace('www.', '')
        return dom if dom else 'DESCONHECIDO'
    except Exception:
        return 'DESCONHECIDO'


if STATUS_DOWNLOAD_FB == 'falhou':
    raise RuntimeError('Download falhou. Siga as instrucoes manuais acima.')

FULL_TEXTS_DIR = _encontrar_full_texts(FAKEBR_EXTRACT_DIR)
if FULL_TEXTS_DIR is None:
    raise FileNotFoundError(
        f'Nao foi possivel localizar pasta full_texts em {FAKEBR_EXTRACT_DIR}.\n'
        'Verifique se o ZIP foi extraido corretamente.'
    )

FAKE_DIR      = FULL_TEXTS_DIR / 'fake'
TRUE_DIR      = FULL_TEXTS_DIR / 'true'
FAKE_META_DIR = FULL_TEXTS_DIR / 'fake-meta-information'
TRUE_META_DIR = FULL_TEXTS_DIR / 'true-meta-information'

fake_files = sorted(FAKE_DIR.glob('*.txt'), key=lambda p: int(p.stem))
true_files = sorted(TRUE_DIR.glob('*.txt'), key=lambda p: int(p.stem))

print(f'full_texts dir         : {FULL_TEXTS_DIR}')
print(f'Arquivos fake          : {len(fake_files)}')
print(f'Arquivos true          : {len(true_files)}')
print(f'Metadados fake         : {len(list(FAKE_META_DIR.glob("*.txt")))}')
print(f'Metadados true         : {len(list(TRUE_META_DIR.glob("*.txt")))}')
print()

# Leitura com metadados
registros_fakebr = []

for path in fake_files:
    texto = _ler_txt(path).strip()
    if not texto:
        continue
    meta_path = FAKE_META_DIR / f'{path.stem}-meta.txt'
    meta = _parse_meta(meta_path) if meta_path.exists() else {}
    url = meta.get('url', '')
    registros_fakebr.append({
        'texto_principal': texto,
        'label': 0,
        'label_detalhe': 'FAKEBR_FALSO',
        'url_origem': url,
        'data_publicacao': meta.get('data', ''),
        'portal_origem_meta': meta.get('categoria', ''),
        '_url': url,
    })

for path in true_files:
    texto = _ler_txt(path).strip()
    if not texto:
        continue
    meta_path = TRUE_META_DIR / f'{path.stem}-meta.txt'
    meta = _parse_meta(meta_path) if meta_path.exists() else {}
    url = meta.get('url', '')
    registros_fakebr.append({
        'texto_principal': texto,
        'label': 1,
        'label_detalhe': 'FAKEBR_REAL',
        'url_origem': url,
        'data_publicacao': meta.get('data', ''),
        'portal_origem_meta': meta.get('categoria', ''),
        '_url': url,
    })

n_fake_lidos = sum(1 for r in registros_fakebr if r['label'] == 0)
n_true_lidos = sum(1 for r in registros_fakebr if r['label'] == 1)
df_fakebr_raw = pd.DataFrame(registros_fakebr)
df_fakebr_raw['tamanho_chars'] = df_fakebr_raw['texto_principal'].str.len()

print(f'Total de registros lidos: {len(df_fakebr_raw)}')
print(f'  label=0 (fake): {n_fake_lidos}')
print(f'  label=1 (real): {n_true_lidos}')
print()
print('Diagnostico de tamanho:')
for lbl in [0, 1]:
    s = df_fakebr_raw[df_fakebr_raw['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): media={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max()}')
print()
print('Percentis de tamanho:')
for lbl in [0, 1]:
    s = df_fakebr_raw[df_fakebr_raw['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}):')
    for p in [10, 25, 50, 75, 90, 95]:
        print(f'    P{p:2d}: {s.quantile(p/100):.0f}')
print()
n_com_url = (df_fakebr_raw['url_origem'].fillna('').str.strip() != '').sum()
n_com_data = (df_fakebr_raw['data_publicacao'].fillna('').str.strip().replace('', np.nan).notna()).sum()
print(f'Registros com URL      : {n_com_url} / {len(df_fakebr_raw)}')
print(f'Registros com data     : {n_com_data} / {len(df_fakebr_raw)}')
print()
print('Exemplos fake (2 primeiros):')
for _, row in df_fakebr_raw[df_fakebr_raw['label']==0].head(2).iterrows():
    print(f'  ({row["tamanho_chars"]} chars | {row["url_origem"][:60]}): {str(row["texto_principal"])[:100]}...')
print()
print('Exemplos real (2 primeiros):')
for _, row in df_fakebr_raw[df_fakebr_raw['label']==1].head(2).iterrows():
    print(f'  ({row["tamanho_chars"]} chars | {row["url_origem"][:60]}): {str(row["texto_principal"])[:100]}...')


full_texts dir         : C:\Users\offan\Desktop\ml-checkai\dados\pipeline_datasets_academicos\raw\fakebr\fakebr_corpus_raw\Fake.br-Corpus-master\full_texts
Arquivos fake          : 3600
Arquivos true          : 3600
Metadados fake         : 3600
Metadados true         : 3600



Total de registros lidos: 7200
  label=0 (fake): 3600
  label=1 (real): 3600

Diagnostico de tamanho:
  label=0 (fake): media=1124 | mediana=956 | min=45 | max=13280
  label=1 (real): media=6673 | mediana=5581 | min=114 | max=46084

Percentis de tamanho:
  label=0 (fake):
    P10: 510
    P25: 696
    P50: 956
    P75: 1355
    P90: 1891
    P95: 2313
  label=1 (real):
    P10: 2690
    P25: 3872
    P50: 5581
    P75: 8592
    P90: 12022
    P95: 13758

Registros com URL      : 7174 / 7200
Registros com data     : 5879 / 7200

Exemplos fake (2 primeiros):
  (1035 chars | https://ceticismopolitico.com/2017/11/30/katia-abreu-diz-que): Kátia Abreu diz que vai colocar sua expulsão em uma moldura, mas não para de reclamar.	

A senadora ...
  (1685 chars | https://ceticismopolitico.com/2017/11/28/blog-esquerdista-da): Blog esquerdista dá a entender que reclamar de dedada no futebol é sinal de homofobia.

Um texto de ...

Exemplos real (2 primeiros):
  (915 chars | http://politica.estadao.co

In [12]:
# Converter tipo e limpar
df_fakebr_raw['texto_principal'] = df_fakebr_raw['texto_principal'].astype(str).str.strip()
df_fakebr_raw['tamanho_chars']   = df_fakebr_raw['texto_principal'].str.len()

# Inferir portal a partir da URL dos metadados
df_fakebr_raw['portal_origem'] = df_fakebr_raw['_url'].apply(_inferir_portal_de_url)

# Preencher colunas do schema
df_fakebr_raw['pipeline_origem']    = PIPELINE_ORIGEM_FB
df_fakebr_raw['origem_texto']       = ORIGEM_TEXTO_FB
df_fakebr_raw['origem_qualidade']   = ORIGEM_QUALIDADE_FB
df_fakebr_raw['fonte_dataset']      = FONTE_DATASET_FB
df_fakebr_raw['referencia_dataset'] = REFERENCIA_ACADEMICA_FB

# Gerar IDs sequenciais
df_fakebr_raw = df_fakebr_raw.reset_index(drop=True)
df_fakebr_raw['id_registro'] = [f'{PREFIXO_ID_FB}_{i+1:06d}' for i in range(len(df_fakebr_raw))]
assert df_fakebr_raw['id_registro'].is_unique, 'id_registro com duplicatas'

# Selecionar colunas do schema (sem _url e portal_origem_meta)
df_fakebr_schema = df_fakebr_raw[COLUNAS_SCHEMA].copy()

# Salvar raw padronizado com timestamp
TS_FB = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
ARQUIVO_SAIDA_FB = RAW_FAKEBR_DIR / f'fakebr_raw_padronizado_{TS_FB}.csv'
df_fakebr_schema.to_csv(ARQUIVO_SAIDA_FB, index=False, encoding='utf-8')

print('=== Schema e saida ===')
print(f'Shape: {df_fakebr_schema.shape}')
print()
print('Distribuicao por label:')
for lbl in [0, 1]:
    cnt = (df_fakebr_schema['label'] == lbl).sum()
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): {cnt}')
print()
print('Estatisticas de tamanho por label:')
for lbl in [0, 1]:
    s = df_fakebr_schema[df_fakebr_schema['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): media={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max()}')
print()
print('Portal de origem (top 10):')
print(df_fakebr_schema['portal_origem'].value_counts().head(10).to_string())
print()
n_url = (df_fakebr_schema['url_origem'].fillna('').str.strip() != '').sum()
n_data = (df_fakebr_schema['data_publicacao'].fillna('').str.strip() != '').sum()
print(f'Registros com URL : {n_url} / {len(df_fakebr_schema)}')
print(f'Registros com data: {n_data} / {len(df_fakebr_schema)}')
print()
print(f'Arquivo salvo: {ARQUIVO_SAIDA_FB.name}')
print(f'Tamanho      : {ARQUIVO_SAIDA_FB.stat().st_size:,} bytes')


=== Schema e saida ===
Shape: (7200, 13)

Distribuicao por label:
  label=0 (fake): 3600
  label=1 (real): 3600

Estatisticas de tamanho por label:
  label=0 (fake): media=1124 | mediana=956 | min=45 | max=13280
  label=1 (real): media=6673 | mediana=5581 | min=114 | max=46084

Portal de origem (top 10):
portal_origem
diariodobrasil.org        3333
G1 Globo                  2275
Estadão                   1202
afolhabrasil.com.br        173
Folha de S.Paulo            95
thejornalbrasil.com.br      66
DESCONHECIDO                26
CeticismoPolítico           16
topfivetv.com                7
Correio                      5

Registros com URL : 7174 / 7200
Registros com data: 5879 / 7200

Arquivo salvo: fakebr_raw_padronizado_2026-05-24_19-58-38.csv
Tamanho      : 32,915,042 bytes


In [13]:
SEP_FB = '=' * 65
AVISO_FB = '!' * 65
print(SEP_FB)
print('RELATORIO — importacao Fake.Br Corpus')
print(SEP_FB)
print(f'Dataset         : {NOME_DATASET_FB}')
print(f'Pipeline        : {PIPELINE_ORIGEM_FB}')
print(f'Origem qualidade: {ORIGEM_QUALIDADE_FB}')
print(f'Referencia      : {REFERENCIA_ACADEMICA_FB}')
print()
print(f'Arquivos fake lidos  : {n_fake_lidos}')
print(f'Arquivos true lidos  : {n_true_lidos}')
print(f'Registros no schema  : {len(df_fakebr_schema)}')
print()
print('[Tamanho por label]')
for lbl in [0, 1]:
    s = df_fakebr_schema[df_fakebr_schema['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}): media={s.mean():.0f} | mediana={s.median():.0f} | min={s.min()} | max={s.max()}')
print()
print('[Percentis de tamanho por label]')
for lbl in [0, 1]:
    s = df_fakebr_schema[df_fakebr_schema['label'] == lbl]['tamanho_chars']
    nome = 'fake' if lbl == 0 else 'real'
    print(f'  label={lbl} ({nome}):')
    for p in [10, 25, 50, 75, 90, 95]:
        print(f'    P{p:2d}: {s.quantile(p/100):.0f}')
print()
print('[Portal de origem]')
print(df_fakebr_schema['portal_origem'].value_counts().to_string())
print()
print(f'Arquivo salvo: {ARQUIVO_SAIDA_FB}')
print()
print(AVISO_FB)
print('AVISO: Este arquivo NAO esta curado para treino.')
print('Proxima etapa: curadoria em src/curadoria_datasets_academicos.ipynb')
print(AVISO_FB)
print(SEP_FB)


RELATORIO — importacao Fake.Br Corpus
Dataset         : Fake.Br Corpus
Pipeline        : datasets_academicos
Origem qualidade: ROTULO_ACADEMICO
Referencia      : Monteiro, R. A. et al. Contributions to the Study of Fake News in Portuguese: New Corpus and Automatic Detection Results. In: Proceedings of the 13th International Conference on Computational Processing of Portuguese (PROPOR), 2018. Disponivel em: https://github.com/roneysco/Fake.br-Corpus

Arquivos fake lidos  : 3600
Arquivos true lidos  : 3600
Registros no schema  : 7200

[Tamanho por label]
  label=0 (fake): media=1124 | mediana=956 | min=45 | max=13280
  label=1 (real): media=6673 | mediana=5581 | min=114 | max=46084

[Percentis de tamanho por label]
  label=0 (fake):
    P10: 510
    P25: 696
    P50: 956
    P75: 1355
    P90: 1891
    P95: 2313
  label=1 (real):
    P10: 2690
    P25: 3872
    P50: 5581
    P75: 8592
    P90: 12022
    P95: 13758

[Portal de origem]
portal_origem
diariodobrasil.org                 3333
